# Data Transformations with PySpark & Spark SQL

## What's covered

- The **bronze → silver** shape — what changes between layers and why
- **Reading bronze**, **cleaning nulls**, **standardising types** — the daily blocking-and-tackling
- **Column operations** — `select`, `withColumn`, `withColumnRenamed`, `drop`, casting
- **Row operations** — `filter` / `where`, `distinct`, `dropDuplicates(subset)`, `dropna`
- **Joins** — `inner`, `left`, `right`, `full`, `left_semi`, `left_anti`, `cross`, multi-key joins
- **Broadcast joins** — when, how, and the `spark.sql.autoBroadcastJoinThreshold` knob
- **Unions** — `union` vs. `unionAll` vs. `unionByName`
- **Array handling** — `split`, `explode`, `posexplode`, `array_contains`
- **Deduplication** — `distinct`, `dropDuplicates`, and SCD-style "keep latest"
- **Aggregates** — `count`, `approx_count_distinct`, `mean`, `sum`, `summary`, `agg`
- **`groupBy` + window functions** — the patterns the bank's gold layer needs
- **Tuning knobs** — `spark.sql.shuffle.partitions`, `spark.default.parallelism`, `spark.executor.memory`, `spark.driver.memory`, `spark.sql.autoBroadcastJoinThreshold`
- **Writing silver** — `saveAsTable`, partitioning vs. liquid clustering, `MERGE INTO` for upserts

## Reference domain

We continue with the Fintech bank. The pipeline shape is the same one we built ingestion for in notebook 03:

```text
  bronze.card_transactions_raw      ──┐
  bronze.bank_accounts_raw          ──┼──► silver.* (clean, conformed, deduplicated)
  bronze.loan_accounts_raw          ──┘
  bronze.customers_raw
```

Silver tables share one rule: **`customer_id` is `STRING` everywhere**, even though raw sources use mixed types. That standardisation is the job of the transformations in this notebook.

## Bronze → silver — what changes between layers

Bronze is *raw and append-only*. Silver is *clean, conformed, deduplicated, and stable*. The transformation between them is what this notebook is about.

Six things almost every bronze-to-silver step does:

1. **Drop or replace nulls** in required columns (an `INSERT` with a null `customer_id` should never reach silver).
2. **Cast types** — bronze often has everything as STRING; silver pins DECIMAL, TIMESTAMP, BOOLEAN.
3. **Normalise values** — uppercase merchant categories, trim whitespace, standardise country codes to ISO-3166.
4. **Deduplicate** — the same transaction may arrive twice if the upstream retried; pick one row per natural key.
5. **Flatten nested structs** that bronze landed verbatim from JSON.
6. **Join in conformed dimensions** — the silver `card_transactions` row carries the customer's city directly, not just a foreign key.

What silver does **not** do: pre-aggregate. Sums-by-day, customer-360 rollups, monthly cohorts — those belong in gold. Silver stays at the *grain of one event per row*, just cleaner.

## Reading bronze

Two equivalent ways — Python and SQL — that the exam may show either side of:

```python
 df = spark.read.table("fintech_dev.bronze.card_transactions_raw")
```

```sql
 SELECT * FROM fintech_dev.bronze.card_transactions_raw
```

Both return a lazy DataFrame / SQL result. Reading a Delta table never reads bytes until an action runs. The Catalyst optimiser will push column pruning, predicate pushdown, and partition pruning into the read.

**Reading a specific Delta version** — useful for *reproducible silver*:

```python
 df = (spark.read.format("delta")
     .option("versionAsOf", 12)
     .table("fintech_dev.bronze.card_transactions_raw"))
```

**Streaming read** — when silver should update as bronze updates:

```python
 stream = spark.readStream.table("fintech_dev.bronze.card_transactions_raw")
```

## Cleaning nulls

Three operations cover ninety percent of what you'll do.

**Drop rows with nulls in required columns** — `dropna(subset=[...])`:

```python
 clean = df.dropna(subset=["transaction_id", "customer_id", "amount"])
```

**Replace nulls with a default** — `fillna({...})`:

```python
 clean = df.fillna({"merchant_category": "unknown", "is_flagged": False})
```

**Conditional replacement** — `when()` + `otherwise()`:

```python
 from pyspark.sql.functions import when, col
 clean = df.withColumn(
     "status",
     when(col("status").isNull(), "pending").otherwise(col("status"))
 )
```

**Tip the exam likes:** `dropna(how="all")` drops only rows where *every* column is null; `dropna(how="any")` (the default) drops if *any* column is null. Use `subset=` to scope it.

## Standardising types

Bronze often has everything as STRING — that's by design, because the schema-on-read step shouldn't lose data. Silver pins real types.

```python
 from pyspark.sql.functions import col, to_timestamp
 silver = (df
     .withColumn("amount",         col("amount").cast("decimal(18,2)"))
     .withColumn("is_flagged",     col("is_flagged").cast("boolean"))
     .withColumn("transaction_at", to_timestamp("transaction_at"))
 )
```

**The casting pitfall** — a string that won't parse silently becomes `NULL`, *not* an error. Defend against it:

- Land bad rows in a quarantine table (`silver.card_transactions_quarantine`) and alert on row count.
- Or use `try_cast` (available in Spark SQL) which is identical to `CAST` but explicit about its null-on-failure behaviour:

```sql
 SELECT transaction_id,
        try_cast(amount AS DECIMAL(18,2)) AS amount
 FROM bronze.card_transactions_raw
```

## Column operations

Four operations and one principle.

- **`select(...)`** — pick / reorder / expression-evaluate columns. Returns a new DataFrame.
- **`withColumn('new', expr)`** — add or replace one column. Cheapest way to derive.
- **`withColumnRenamed('old', 'new')`** — rename in place. Useful right before a join when both sides share a name.
- **`drop('col1', 'col2', ...)`** — remove columns.

**The principle: every transformation returns a *new* DataFrame.** Spark DataFrames are immutable. `df.withColumn(...)` doesn't change `df`; it returns a new DataFrame that you bind to a variable.

**Casting via expression**, instead of `withColumn` + `cast` chains:

```python
 silver = df.selectExpr(
     "transaction_id",
     "customer_id",
     "CAST(amount AS DECIMAL(18,2)) AS amount",
     "to_timestamp(transaction_at) AS transaction_at"
 )
```

## Row operations

**`filter` / `where`** are aliases — pick one and stay consistent. Both accept a boolean column or a SQL string:

```python
 df.filter(col("amount") > 1000)
 df.where("status = 'approved' AND is_flagged = false")
```

**`distinct()`** — drop full-row duplicates. Cheap if the rows really are identical.

**`dropDuplicates(subset=[...])`** — drop rows where the listed columns are duplicates. The row that survives is *unspecified* unless you order first.

**SCD-style "keep latest" deduplication** — the pattern the bank's silver layer uses to handle retried upstream sends:

```python
 from pyspark.sql.window import Window
 from pyspark.sql.functions import row_number, col

 w = Window.partitionBy("transaction_id").orderBy(col("ingested_at").desc())
 deduped = (df.withColumn("rn", row_number().over(w))
              .filter("rn = 1")
              .drop("rn"))
```

**Why the exam loves the window trick:** it's deterministic. `dropDuplicates` is not — for two rows with the same transaction_id, which one survives depends on partition order.

## Joins — the seven types

Spark supports the full SQL join family plus two Spark-specific shapes (`semi`, `anti`). Memorise this table:

| Join type | Returns |
|---|---|
| `inner` | Rows where the key matches on both sides |
| `left` (a.k.a. `left_outer`) | All rows from left; nulls on right when no match |
| `right` (a.k.a. `right_outer`) | All rows from right; nulls on left when no match |
| `full` (a.k.a. `full_outer`, `outer`) | All rows from both sides; nulls on the missing side |
| `left_semi` | Rows from left **that have a match** on the right — *no columns from right* |
| `left_anti` | Rows from left **that do NOT match** on the right |
| `cross` | Cartesian product. Rarely the right answer. |

Two API shapes:

```python
 # Same key column on both sides — pass a string
 df.join(customers, on="customer_id", how="inner")

 # Different names — pass a column expression
 df.join(customers, df.cust_id == customers.customer_id, how="left")

 # Multi-key — pass a list of column names
 df.join(other, on=["customer_id", "account_id"], how="inner")
```

**`left_semi` is the exam favourite** for the question "give me customers who have at least one transaction" — `select` on the customers DataFrame would scan every row; `left_semi` short-circuits at the first match per key.

## Broadcast joins

A normal join shuffles both sides by the join key — expensive for large tables. A **broadcast join** ships the smaller side to *every executor*, eliminating the shuffle.

**Spark auto-broadcasts** when one side's estimated size is below `spark.sql.autoBroadcastJoinThreshold` (default 10 MB, often tuned to 100 MB or more). For the bank, joining 50 billion `card_transactions` rows against a 5 MB `merchant_categories` lookup auto-broadcasts.

**Force a broadcast** when stats are stale or you know better than the optimiser:

```python
 from pyspark.sql.functions import broadcast
 df.join(broadcast(small_df), on="merchant_category", how="left")
```

```sql
 SELECT /*+ BROADCAST(c) */ t.*, c.category_group
 FROM card_transactions t
 LEFT JOIN merchant_categories c ON t.merchant_category = c.code
```

**When it backfires:** broadcasting a side larger than driver memory blows the driver. Cap the broadcast threshold at the same order as the largest small table you actually have.

**Adaptive Query Execution** (AQE — covered in 08) can also promote a sort-merge join to a broadcast join *at runtime* once it sees real partition stats, even if planner statistics said otherwise.

## Unions — `union`, `unionAll`, `unionByName`

Three operations that sound similar and behave differently.

- **`union(other)`** — concatenates rows. Matches columns by **position**. Both DataFrames must have the same number of columns in the same order. Does **NOT** deduplicate (despite the SQL UNION default). Same as `unionAll`.
- **`unionAll(other)`** — alias for `union`. Deprecated name but still works.
- **`unionByName(other, allowMissingColumns=False)`** — matches columns by **name**, not position. Safer in production. With `allowMissingColumns=True`, missing columns are filled with NULL.

**The exam trap:** developers expect `union` to deduplicate (because SQL `UNION` does). Spark's `union` is closer to SQL's `UNION ALL`. To deduplicate, chain `.distinct()`:

```python
 all_txns = jan.union(feb).distinct()
```

Prefer **`unionByName`** in any code that survives schema changes.

## Splitting and exploding arrays

Three primitives the exam tests directly.

**`split(col, pattern)`** — turns a string into an array. Useful for tag columns: `'travel,food,fuel'` → `['travel','food','fuel']`.

**`explode(arr)`** — one row per array element. The classic pattern for line-items / events arrays.

**`posexplode(arr)`** — same but also emits the position index (`pos`).

```python
 from pyspark.sql.functions import split, explode, posexplode

 # 'grocery,fuel' → ['grocery','fuel'] → two rows
 df.withColumn("tags", split("merchant_tags", ",")) \
   .withColumn("tag", explode("tags")) \
   .drop("tags")
```

**`array_contains(arr, value)`** — the predicate version, used in `WHERE` instead of materialising every row:

```sql
 WHERE array_contains(tags, 'travel')
```

**Inverse — collecting back into an array:** `collect_list(col)` (keeps duplicates, order-undefined) and `collect_set(col)` (deduplicates) inside a `groupBy`.

## Aggregates the exam tests

Five aggregates with specific exam-relevant behaviour.

- **`count('*')`** vs. **`count('col')`** — `count('*')` counts every row; `count('col')` skips nulls. The bank's "how many transactions did this customer make" uses `count('*')`; "how many transactions had a merchant category populated" uses `count('merchant_category')`.
- **`approx_count_distinct(col, rsd=0.05)`** — HyperLogLog-based approximate distinct count. Orders of magnitude cheaper than `countDistinct` on billion-row tables. The exam expects you to pick it when *approximate is OK*.
- **`mean(col)`** / **`avg(col)`** — same function, two names.
- **`sum(col)`**, **`min(col)`**, **`max(col)`** — null-skipping by default.
- **`summary(*stats)`** — a DataFrame method that emits count, mean, stddev, min, max, and percentiles in one call. Used for ad-hoc data profiling.

**The `agg` pattern** for several aggregates at once:

```python
 from pyspark.sql.functions import count, approx_count_distinct, mean, sum as _sum

 (silver.groupBy("customer_id")
        .agg(count("*").alias("txn_count"),
             approx_count_distinct("merchant_name").alias("distinct_merchants"),
             _sum("amount").alias("total_spend"),
             mean("amount").alias("avg_spend")))
```

## Window functions — the pattern silver and gold lean on

A window function computes a value *per row* using a window of other rows around it. The classic shape:

```python
 from pyspark.sql.window import Window
 from pyspark.sql.functions import row_number, lag, sum as _sum, col

 # Rank each customer's transactions newest-first — pick #1 for SCD-latest dedup
 w = Window.partitionBy("customer_id").orderBy(col("transaction_at").desc())
 df.withColumn("rn", row_number().over(w))

 # Previous transaction amount per customer
 df.withColumn("prev_amount", lag("amount").over(w))

 # Running total per customer (rangeBetween / rowsBetween)
 w_run = (Window.partitionBy("customer_id")
                .orderBy("transaction_at")
                .rowsBetween(Window.unboundedPreceding, Window.currentRow))
 df.withColumn("running_total", _sum("amount").over(w_run))
```

**The exam tests window functions for**: deduplication (`row_number`), period-over-period calcs (`lag` / `lead`), and running totals (`sum` with `rowsBetween`).

## Tuning knobs — the four the exam asks about

You won't tune in this notebook (notebook 08 owns that), but the exam's Section 3 explicitly calls out four config keys. Know what each controls.

| Config | What it controls | Typical default | When to change |
|---|---|---|---|
| `spark.sql.shuffle.partitions` | Number of partitions Spark uses after a shuffle (joins, aggregations) | 200 | Big jobs: bump up (e.g. 1000+) to reduce per-partition data size and avoid OOM. Small jobs: drop (e.g. 8) to avoid hundreds of empty tasks. AQE coalesces this dynamically when enabled. |
| `spark.default.parallelism` | Default partition count for RDD operations (used by `parallelize`, repartitioning without a number) | `# total cores` | Rarely tuned directly on DataFrame work. Mostly an RDD knob. |
| `spark.executor.memory` / `spark.driver.memory` | Heap per executor / driver JVM | Cluster-dependent | OOM in executor tasks → raise executor memory or split work. Collect-to-driver crashes → raise driver memory. |
| `spark.sql.autoBroadcastJoinThreshold` | Max size of a side that Spark will auto-broadcast | 10 MB | Bump to 100 MB+ when broadcasting small dimension tables that exceed 10 MB. Set to `-1` to disable auto-broadcast entirely. |

**The way the exam phrases the question:** "a job is shuffling 200 GB across 200 partitions and tasks are spilling to disk — what should you change?" → bump `spark.sql.shuffle.partitions` so each partition is in the 100–200 MB range.

## Writing silver

Two write modes the exam tests:

**Full overwrite** — the silver build is idempotent; re-running rebuilds from bronze.

```python
 (silver.write
        .mode("overwrite")
        .option("overwriteSchema", "true")     # only when schema changed
        .saveAsTable("fintech_dev.silver.card_transactions"))
```

**Upsert with `MERGE INTO`** — silver tracks updates from bronze CDC.

```sql
 MERGE INTO fintech_dev.silver.card_transactions AS t
 USING bronze_updates                            AS s
    ON t.transaction_id = s.transaction_id
 WHEN MATCHED     THEN UPDATE SET *
 WHEN NOT MATCHED THEN INSERT *
```

**Layout decisions:**

- For new tables: **Liquid Clustering** on the natural query keys (`CLUSTER BY (customer_id, transaction_at)`) — covered in notebook 02.
- For legacy partitioned tables: `partitionBy("transaction_date")` — but new tables should prefer Liquid Clustering.
- **Enable Predictive Optimization** at the catalog level (notebook 02) so silver stays compact without a maintenance job.

## Hands-on — bronze.card_transactions_raw → silver.card_transactions

We re-create the bronze table from notebook 03 with a tiny inline dataset (so this notebook is self-contained), then run the full bronze-to-silver pipeline: clean nulls, cast types, flatten, deduplicate, and write silver.

In [ ]:
spark.sql("USE CATALOG fintech_dev")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [ ]:
# A tiny bronze table — strings throughout, nulls present, one duplicate, one nested merchant.
from pyspark.sql import Row
from datetime import datetime

bronze_rows = [
    Row(transaction_id="T100", customer_id="C001", amount="1500.00",  merchant_name="BigBasket",  merchant_category="grocery", transaction_at="2024-01-05T10:30:00", is_flagged="false", ingested_at=datetime(2024,1,5,10,31)),
    Row(transaction_id="T101", customer_id="C002", amount="8200.00",  merchant_name="MakeMyTrip", merchant_category="travel",  transaction_at="2024-01-06T14:10:00", is_flagged="false", ingested_at=datetime(2024,1,6,14,11)),
    Row(transaction_id="T101", customer_id="C002", amount="8200.00",  merchant_name="Cleartrip",  merchant_category="travel",  transaction_at="2024-01-06T14:10:00", is_flagged="false", ingested_at=datetime(2024,1,6,15,00)),  # late correction
    Row(transaction_id="T102", customer_id=None,    amount="3000.00",  merchant_name="Shell",      merchant_category="fuel",    transaction_at="2024-01-07T09:00:00", is_flagged="false", ingested_at=datetime(2024,1,7,9,1)),    # missing customer_id
    Row(transaction_id="T103", customer_id="C001", amount="45000.00", merchant_name="Unknown",    merchant_category=None,      transaction_at="2024-01-05T23:55:00", is_flagged="true",  ingested_at=datetime(2024,1,5,23,56)),
]
spark.createDataFrame(bronze_rows).write.mode("overwrite").saveAsTable("bronze.card_transactions_raw")
spark.sql("SELECT * FROM bronze.card_transactions_raw").show(truncate=False)

In [ ]:
# Step 1+2 — drop rows missing customer_id, fill missing merchant_category, cast types.
from pyspark.sql.functions import col, to_timestamp

df = spark.read.table("bronze.card_transactions_raw")

cleaned = (df
    .dropna(subset=["transaction_id", "customer_id", "amount"])
    .fillna({"merchant_category": "unknown"})
    .withColumn("amount",         col("amount").cast("decimal(18,2)"))
    .withColumn("is_flagged",     col("is_flagged").cast("boolean"))
    .withColumn("transaction_at", to_timestamp("transaction_at"))
)
cleaned.printSchema()
cleaned.show(truncate=False)

In [ ]:
# Step 4 — deterministic dedup. For repeats of the same transaction_id keep the LATEST ingest.
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

w = Window.partitionBy("transaction_id").orderBy(col("ingested_at").desc())
deduped = (cleaned.withColumn("rn", row_number().over(w))
                  .filter("rn = 1")
                  .drop("rn"))

print("Before dedup:", cleaned.count())
print("After dedup :", deduped.count())
deduped.select("transaction_id", "merchant_name", "ingested_at").show(truncate=False)

In [ ]:
# Step 6 — join in a small dimension. broadcast() forces the small side onto every executor.
from pyspark.sql.functions import broadcast

merchant_groups = spark.createDataFrame([
    ("grocery", "essentials"),
    ("fuel",    "essentials"),
    ("travel",  "discretionary"),
    ("unknown", "unknown"),
], ["merchant_category", "category_group"])

enriched = deduped.join(broadcast(merchant_groups), on="merchant_category", how="left")
enriched.select("transaction_id", "merchant_category", "category_group", "amount").show(truncate=False)

In [ ]:
# Aggregates — count, approx_count_distinct, sum, mean — grouped by customer.
from pyspark.sql.functions import count, approx_count_distinct, sum as _sum, mean

(enriched.groupBy("customer_id")
         .agg(count("*").alias("txn_count"),
              approx_count_distinct("merchant_name").alias("distinct_merchants"),
              _sum("amount").alias("total_spend"),
              mean("amount").alias("avg_spend"))
         .orderBy("customer_id")
         .show(truncate=False))

In [ ]:
# Step 7 — write silver. Liquid Clustering on customer_id + transaction_at is the modern default.
(enriched
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .clusterBy("customer_id", "transaction_at")
    .saveAsTable("silver.card_transactions"))

spark.sql("SELECT COUNT(*) AS silver_rows FROM silver.card_transactions").show()
spark.sql("DESCRIBE DETAIL silver.card_transactions").select("format", "clusteringColumns", "numFiles", "sizeInBytes").show(truncate=False)

In [ ]:
# Inspect the tuning knobs the exam tests — and try changing one.
print("spark.sql.shuffle.partitions          =", spark.conf.get("spark.sql.shuffle.partitions"))
print("spark.default.parallelism             =", spark.sparkContext.defaultParallelism)
print("spark.sql.autoBroadcastJoinThreshold  =", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

# Bump the broadcast threshold to 100 MB for the rest of this session.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 100 * 1024 * 1024)
print("After change                          =", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

## Section 3 (part 1) self-check

**1.** Which join returns *only the rows of the left DataFrame that have at least one match on the right*, and brings *no columns* from the right?

- A. `inner`
- B. `left`
- C. `left_semi`
- D. `left_anti`

**2.** A 30 MB dimension table is being shuffle-joined to a 5 TB fact table and the job is slow. Which change is most likely to help?

- A. Lower `spark.sql.shuffle.partitions` to 8
- B. Raise `spark.sql.autoBroadcastJoinThreshold` past 30 MB (or wrap the dimension in `broadcast()`)
- C. Convert to a `cross` join
- D. Disable AQE

**3.** `unionByName(other, allowMissingColumns=True)` does what?

- A. Matches columns by position
- B. Matches columns by name and fills missing columns with NULL
- C. Deduplicates rows
- D. Broadcasts the smaller side

**4.** The bank wants an approximate count of distinct merchants across 50 billion card transactions. Which function is the cheapest correct answer?

- A. `count('merchant_name')`
- B. `countDistinct('merchant_name')`
- C. `approx_count_distinct('merchant_name')`
- D. `collect_set('merchant_name')` then `size`

**5.** A bronze table has duplicates of the same `transaction_id` from upstream retries. The team wants silver to keep the row with the latest `ingested_at`. Which approach is **deterministic**?

- A. `dropDuplicates(['transaction_id'])`
- B. `distinct()`
- C. `row_number()` over partition `transaction_id` ordered by `ingested_at DESC`, keep `rn = 1`
- D. `groupBy('transaction_id').count()`

<details><summary>Show answers</summary>

1. **C** — `left_semi` returns left rows with a match and no right columns.
2. **B** — broadcast the small dimension.
3. **B** — name-based union with NULL fill.
4. **C** — HyperLogLog approximate distinct count.
5. **C** — window + `row_number()` is deterministic; `dropDuplicates` is not.

</details>

## Summary

- Bronze is raw and append-only; silver is **clean, conformed, deduplicated, and stable**. Same grain, just trustworthy.
- **Cleaning nulls**: `dropna(subset=[...])`, `fillna({...})`, `when().otherwise()`.
- **Casting**: bronze STRING → silver typed; bad parses become NULL, not errors — quarantine or `try_cast` to defend.
- **Columns**: `select`, `withColumn`, `withColumnRenamed`, `drop`. DataFrames are immutable.
- **Rows**: `filter` / `where`, `distinct`, `dropDuplicates(subset)`. For latest-wins dedup, use a **window + `row_number()`**.
- **Joins**: seven types. `left_semi` is the exam favourite for "customers who have at least one transaction". Multi-key joins pass a list. **Broadcast** the small side when one fits in memory.
- **Unions**: `union` matches by position and doesn't dedupe. Prefer **`unionByName`**.
- **Arrays**: `split`, `explode`, `posexplode`, `array_contains`, `collect_list` / `collect_set`.
- **Aggregates**: `count('*')` vs `count('col')`, `approx_count_distinct` for billion-row distinct counts.
- **Windows**: `row_number` for dedup, `lag` / `lead` for period-over-period, `sum` with `rowsBetween` for running totals.
- **Tuning knobs** to recognise: `spark.sql.shuffle.partitions`, `spark.default.parallelism`, `spark.executor.memory` / `spark.driver.memory`, `spark.sql.autoBroadcastJoinThreshold`.
- **Write silver** with overwrite + `clusterBy(...)` for new tables; **`MERGE INTO`** for CDC.

Next up: **notebook 05 — Modeling: Medallion, MVs, Streaming Tables, Lakeflow Spark Declarative Pipelines** — how to model gold for BI / ML, the difference between tables, views, materialized views, and streaming tables, and how Lakeflow Spark Declarative Pipelines (formerly DLT) declares all of this with expectations for data quality.